# Lektion 04: Chatbot mit Gedächtnis



| | |
|---|---|
| **Zeit** | ~60 Minuten |
| **Schwierigkeit** | Leicht |
| **Voraussetzungen** | Lektion 03 |
| **Du baust** | Einen funktionierenden Chatbot, der sich an das Gespräch erinnert |

## 🎯 Nach dieser Lektion kannst du:

- erklären, warum LLMs **gedächtnislos** sind – und was das für deine Architektur bedeutet

- einen Gesprächsverlauf als Nachrichten-Liste pflegen

- eine Budget-Bremse einbauen (Historie begrenzen)

- mit Temperature experimentieren und die Wirkung dokumentieren

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🧠 Konzept: Das Gedächtnis-Geheimnis

**WARUM:** Jeder API-Aufruf ist ein **isolierter Moment**. Das Modell erinnert sich an NICHTS zwischen zwei Aufrufen – dein Chatbot aus Lektion 03 hat "vergessen", was du eine Sekunde vorher gefragt hast.

**DAS Geheimnis:** Chatbots mit "Gedächtnis" schicken einfach **die ganze bisherige Liste** wieder mit. **Metapher:** Der Assistent hat Goldfisch-Gedächtnis. **Du** als Programmierer bist die Lösung: Du führst das Protokoll (die `messages`-Liste) und legst es bei jeder Frage **komplett wieder auf den Tisch**.

Schau genau hin – das ist buchstäblich eine Liste aus Lektion 01:

In [2]:
from lernakademie.llm_client import get_api_client, get_model_name

client = get_api_client()
verlauf = [
    {"role": "system", "content": "Du bist ein freundlicher Lern-Coach fuer Python-Anfaenger."},
]

def sage(text):
    """Ein Gespraechszug: Frage anhängen, senden, Antwort merken."""
    verlauf.append({"role": "user", "content": text})
    antwort = client.chat.complete(model=get_model_name(), messages=verlauf)
    inhalt = antwort.choices[0].message.content
    verlauf.append({"role": "assistant", "content": inhalt})
    return inhalt

sage("Hallo! Ich heisse Alex und lerne Python.")
antwort = sage("Wie heisse ich noch mal?")   # <-- Erinnerungstest!
print("Antwort auf die Erinnerungsfrage:")
print(antwort)

Antwort auf die Erinnerungsfrage:
[OFFLINE] Simulierte Antwort (Offline-Modus) auf: „Wie heisse ich noch mal?“


### 🔎 Was sehen wir hier?
Beim zweiten `sage(...)` enthält `verlauf` **vier** Nachrichten: System, Name-Nennung, Modell-Antwort, neue Frage. Das Modell "sieht" den ganzen Tisch – deshalb kann es auf Alex antworten. **Gedächtnis ist hier kein Zauber, sondern Listen-Management.**

In [3]:
for nachricht in verlauf:
    print(f"[{nachricht['role']:9s}] {nachricht['content'][:60]}")

[system   ] Du bist ein freundlicher Lern-Coach fuer Python-Anfaenger.
[user     ] Hallo! Ich heisse Alex und lerne Python.
[assistant] [OFFLINE] Simulierte Antwort (Offline-Modus) auf: „Hallo! Ic
[user     ] Wie heisse ich noch mal?
[assistant] [OFFLINE] Simulierte Antwort (Offline-Modus) auf: „Wie heiss


## 🧠 Konzept: Die Budget-Bremse

**WARUM:** Jede Nachricht kostet Tokens (Lektion 03!) – ein endlos wachsender Verlauf wird **teuer** und irgendwann größer als das Kontextfenster (der Schreibtisch voll).

**WIE:** Wir kürzen die Historie regelmäßig. **Aber:** die System-Nachricht (Index 0) muss IMMER bleiben – sonst vergisst dein Bot, wer er ist!

In [4]:
def kuerze_historie(verlauf, max_nachrichten=20):
    """Behaelt die System-Nachricht (Index 0) und die letzten Nachrichten."""
    if len(verlauf) <= max_nachrichten:
        return verlauf
    return [verlauf[0]] + verlauf[-(max_nachrichten - 1):]

langer_verlauf = [{"role": "system", "content": "Ich bleibe!"}] + \
    [{"role": "user", "content": f"Nachricht {i}"} for i in range(30)]
gekuerzt = kuerze_historie(langer_verlauf)
print("Vorher:", len(langer_verlauf), "| Nachher:", len(gekuerzt))
print("System bleibt:", gekuerzt[0]["content"] == "Ich bleibe!")

Vorher: 31 | Nachher: 20
System bleibt: True


## 🔎 Beispiel: Ein kompletter Chatbot

Jetzt kombinieren wir alles: Verlauf, Senden, Merken, Kürzen – in **einer** sauberen Funktion:

In [5]:
from lernakademie.llm_client import get_api_client, get_model_name

PERSONA = "Du bist ein freundlicher Lern-Coach. Antworte kurz und ohne Fachchinesisch."

def baue_chatbot(persona=PERSONA, max_nachrichten=20):
    client = get_api_client()
    verlauf = [{"role": "system", "content": persona}]

    def sage(text):
        verlauf.append({"role": "user", "content": text})
        antwort = client.chat.complete(model=get_model_name(), messages=verlauf)
        inhalt = antwort.choices[0].message.content
        verlauf.append({"role": "assistant", "content": inhalt})
        if len(verlauf) > max_nachrichten:               # Budget-Bremse
            verlauf[:] = [verlauf[0]] + verlauf[-(max_nachrichten - 1):]
        return inhalt

    return sage

bot = baue_chatbot()
for frage in ("Hallo, ich heisse Kim und arbeite im Marketing.",
              "Wie heisse ich noch mal?",
              "Was ist eine Variable?"):
    print("FRAGE:", frage)
    print("BOT  :", bot(frage)[:120], "\n")

FRAGE: Hallo, ich heisse Kim und arbeite im Marketing.
BOT  : [OFFLINE] Simulierte Antwort (Offline-Modus) auf: „Hallo, ich heisse Kim und arbeite im Marketing.“ 

FRAGE: Wie heisse ich noch mal?
BOT  : [OFFLINE] Simulierte Antwort (Offline-Modus) auf: „Wie heisse ich noch mal?“ 

FRAGE: Was ist eine Variable?
BOT  : [OFFLINE] Simulierte Antwort (Offline-Modus) auf: „Was ist eine Variable?“ 



## ✏️ Deine Aufgabe: Dein eigener Chatbot

Du baust einen Chatbot mit Gedächtnis UND einer sauberen, testbaren Struktur.

**Anforderungen:**

- Schreibe `baue_chatbot(persona=..., max_nachrichten=...)`, das eine Funktion `sage(text)` zurückgibt.

- Jeder Aufruf von `sage` hängt die Frage an den Verlauf an, sendet die **gesamte** Liste und merkt sich die Antwort.

- Der Verlauf wird auf `max_nachrichten` gekürzt – die System-Nachricht bleibt immer.

- Leere Eingaben geben einen Hinweis zurück und landen NICHT im Verlauf.

**Schreibe deinen Code in die folgende Zelle:**

In [6]:
from lernakademie.llm_client import get_api_client, get_model_name

def baue_chatbot(persona="Du bist ein Assistent.", max_nachrichten=20):
    client = get_api_client()
    verlauf = [{"role": "system", "content": persona}]

    def sage(text):
        if not text.strip():
            return "Bitte schreib mir eine echte Nachricht."
        verlauf.append({"role": "user", "content": text})
        antwort = client.chat.complete(model=get_model_name(), messages=verlauf)
        inhalt = antwort.choices[0].message.content
        verlauf.append({"role": "assistant", "content": inhalt})
        if len(verlauf) > max_nachrichten:
            verlauf[:] = [verlauf[0]] + verlauf[-(max_nachrichten - 1):]
        return inhalt

    return sage

bot = baue_chatbot()
print(bot("Hallo, ich heisse Sam!")[:120])

[OFFLINE] Simulierte Antwort (Offline-Modus) auf: „Hallo, ich heisse Sam!“


**💡 Warum diese Loesung funktioniert:**

Der Verlauf lebt in der äußeren Funktion (`closure`) – deshalb "überlebt" er mehrere `sage`-Aufrufe. Das Muster "Daten im Closure + kleine Funktion zurückgeben" begegnet dir in Build 2 erneut (Retriever-Tools!). Die Budget-Bremse greift erst, wenn der Verlauf zu lang wird – die System-Nachricht wird mit `[verlauf[0]]` explizit gerettet.

### ✅ Richtig, wenn:

- [ ] Zwei aufeinanderfolgende `sage`-Aufrufe teilen sich das Gedächtnis (Erinnerungstest!).

- [ ] Der Verlauf wächst nie über `max_nachrichten`.

- [ ] Leere Eingaben craschen nicht.

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [7]:
bot = baue_chatbot(max_nachrichten=10)
bot("Hallo, ich heisse Sam!")
zweiter = bot("Wie heisse ich?")
assert isinstance(zweiter, str) and len(zweiter) > 0
for i in range(15):
    bot(f"Nachricht {i}")
assert True  # kein Absturz nach vielen Nachrichten = Budget-Bremse laeuft
leer = bot("   ")
assert isinstance(leer, str)
print("Alle Prüfungen bestanden! ✅")

Alle Prüfungen bestanden! ✅


<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Kopiere das Muster aus dem Beispiel oben – aber schreib es selbst,和理解 kommt vom Tippen!
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Für den Leerfall: `if not text.strip(): return "..."` ganz am Anfang von `sage`.
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Das Kürzen geht in einem Schritt: `verlauf[:] = [verlauf[0]] + verlauf[-(max_nachrichten-1):]`.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Was würde passieren, wenn du beim Kürzen die System-Nachricht vergisst?

2. Warum kostet ein längeres Gespräch pro Nachricht MEHR als ein kurzes – und wo liegt die Grenze (Stichwort: Kontextfenster)?

## ➡️ Weiter

**🧱 BUILD 1 beginnt – Lektion 05:** `notebooks/03_build_1/05_build1_wissensassistent.ipynb` – dein Chatbot bekommt **echtes Wissen**: Dokumente, eigene Suche, selbst gebauter Prompt. Ganz ohne Framework!